In [5]:
import requests
from bs4 import BeautifulSoup as Bsoup
import numpy as np
import pandas as pd
import urllib.request

#website we are scraping
#url = 'http://mlg.ucd.ie/modules/COMP30760/assignment1/retail/2023Q1-page01.html'
#we have base
base_url = "http://mlg.ucd.ie/modules/COMP30760/assignment1/retail/2023Q1-page"

#list[{with dict inside}] we gonna fill for df. Faster than first initializing df.
all_sale_data = []

#iterate over each URL page
for page in range(1, 16):
    
    #makes sure that i is always 2 digit number so that we find the correct page
    page_num = f'{page:02}'
    
    #changes url to fit with page 
    url = base_url+str(page_num)+'.html'

    #if page fails to load
    try:
        page = requests.get(url)
    except:
        print("error: Failed to retrieve %s url" % url)
    
    #parse html
    soup = Bsoup(page.content, "html.parser")
    
    #find id that contains all the data of each box of data for each sale
    list = soup.find(id="content")
    
    #gets list of all sales
    all_retail_sales = list.find_all('li')
    
    #iterate through single sale
    for retail_sales in all_retail_sales:
        
        #data for single sale in a dictionary
        sale_data = {}
        
        #data for one sale in here
        one_sale = retail_sales.find_all('td')
        
        #iterate by 2 cuz label is every other
        for detail in range(0, len(one_sale)-1, 2):
            
            #define category--> first one
            category = one_sale[detail].text.strip()[0:-1]
            
            #excpetion cuz Customer detials is weird and annoying. 
            #customer details is a category but the data inside it has seperate categories and details

            
            if category == "Customer Details": #different case cuz this is unique

                #get customer details part into list
                customer_details = one_sale[detail+1].get_text(separator=":")
                list_of_customer_details = customer_details.split(':')

                #iterate over list: follows same format as above for previous details
                for i in range(0, len(list_of_customer_details)-1, 2):

                    
                    #names are hard to come up with so kittyCAT(egory)
                    kittyKAT = list_of_customer_details[i].replace(' ', '')
                    #current value
                    value = list_of_customer_details[i+1].strip()


                    
                    #since sometimes they are in mixed order this makes it easier.
                    #and we can fix spacing etc in main categories
                    if kittyKAT == 'ID':
                        sale_data['ID'] = value
                    elif kittyKAT == 'Location':
                        sale_data['Location'] = value
                    elif kittyKAT == 'Gender':
                        sale_data['Gender'] = value
                    elif kittyKAT == 'AgeCategory':
                        sale_data['Age Category'] = value

            
            #if detail is not customer details--> we do normal data and category cuz customer details is only exception
            else:
                #data strip
                data= one_sale[detail+1].text.strip()
                #fix spacing
                data = data.replace(' ', '')

                #we have category from above. We append the sale_data dictionary
                sale_data[category] = data
        #add each sale data to big boy list, for each sale on each page
        all_sale_data.append(sale_data)

#create dataframe with fat list (faster than creating df first)
df = pd.DataFrame(all_sale_data)

#create csv file
df.to_csv('Assignment1_sales_data.csv', index=False)
                
                
 